# 2-3 数组与矩阵的基本操作

NumPy 中的运算需要先分清两件事：**数组的形状**和**运算的含义**。

1. 使用 `reshape()` 调整数组形状
2. 对数组中的对应元素进行运算
3. 使用二维数组进行矩阵运算

> 本节使用 NumPy 的普通数组 `ndarray`。矩阵乘法使用 `@`，不需要创建专门的矩阵对象。

In [ ]:
import numpy as np

## 1. 使用 `reshape()` 改变数组形状

`reshape()` 可以把数组重新排列为指定的形状，**改变形状前后，元素总数必须相同**。

例如，先创建包含 9 个元素的一维数组，再改为 3 行、3 列的二维数组。

In [ ]:
numbers = np.arange(1, 10)
numbers

In [ ]:
a = numbers.reshape(3, 3)
a

`numbers` 的形状为 `(9,)`，`a` 的形状为 `(3, 3)`，元素总数都是 9。

`reshape()` 返回调整形状后的数组；需要保留结果时，应接收返回值。它不保证复制数据，本节只观察形状，不修改共享的元素。

In [ ]:
numbers.shape, a.shape, a.size

也可以使用一个 `-1`，让 NumPy 根据元素总数自动计算该维度的大小。

`reshape(3, -1)` 表示固定为 3 行，列数由 NumPy 计算。9 个元素不能重排成 `(2, 4)`，因为 $9 \neq 2 \times 4$。

In [ ]:
numbers.reshape(3, -1)

## 2. 数组之间的运算

普通 NumPy 数组的 `+`、`-`、`*`、`/` 都是**逐元素运算**。

| 运算符 | 含义 |
|---|---|
| `a + b` | 对应位置的元素相加 |
| `a - b` | 对应位置的元素相减 |
| `a * b` | 对应位置的元素相乘 |
| `a / b` | 对应位置的元素相除 |

先创建两个形状相同的数组：

In [ ]:
a = np.arange(1, 10).reshape(3, 3)
b = np.arange(1, 10).reshape(3, 3)
a, b

### 2.1 加法和减法

In [ ]:
a + b

In [ ]:
a - b

两个数组的对应元素相同，因此相减后得到全 0 数组。

### 2.2 乘法和除法

In [ ]:
a * b

In [ ]:
a / b

`a * b` 得到各个元素的平方；`a / b` 得到全 1 的浮点数数组。

**注意：这里的 `*` 是逐元素乘法，不是矩阵乘法。** 示例中的除数都不为 0。

### 2.3 形状要求与广播

本节主要使用形状相同的数组。更准确地说，逐元素运算要求两个数组的形状**能够广播兼容**，不只是维度数相同。

从形状的末尾开始比较，每一对维度大小必须相同，或其中一个为 `1`；缺少的前置维度按 `1` 处理。

- `(3, 3)` 和 `(3, 3)`：形状相同，可以逐元素运算。
- `(3, 3)` 和 `(3,)`：后一组数可以扩展到每一行。
- `(3, 3)` 和 `(3, 4)`：最后一维的 `3` 和 `4` 不兼容，不能直接相加、相减。

In [ ]:
row = np.array([10, 20, 30])
a + row

结果相当于给 `a` 的每一行分别加上 `[10, 20, 30]`，这就是广播的一个例子。

## 3. 矩阵之间的运算

数学上的矩阵是二维的，可以直接用 NumPy 的**二维数组**表示。

`np.matrix()` 是较早的矩阵专用类型。使用 `np.array()` 或其他数组创建函数生成二维数组后，可以用 `@` 表示矩阵乘法。

| 运算 | 二维数组的写法 | 形状要求 |
|---|---|---|
| 矩阵加法 | `a + b` | 数学上的矩阵加法要求形状相同 |
| 矩阵减法 | `a - b` | 数学上的矩阵减法要求形状相同 |
| 矩阵乘法 | `d @ e` 或 `d.dot(e)` | 左边的列数等于右边的行数 |

前面的 `a + b` 和 `a - b` 也就是这两个同形矩阵的加法和减法。

### 3.1 创建参与矩阵乘法的数组

创建形状为 `(3, 4)` 的 `d` 和形状为 `(4, 3)` 的 `e`：

In [ ]:
d = np.arange(12).reshape(3, 4)
e = np.arange(12).reshape(4, 3)
d

In [ ]:
e

### 3.2 使用 `@` 进行矩阵乘法

`d` 的列数是 4，`e` 的行数也是 4，因此可以相乘。

结果的形状为 `(3, 3)`，即保留左侧矩阵的行数和右侧矩阵的列数。

In [ ]:
product = d @ e
product

In [ ]:
product.shape

结果中的每一个元素，都是左侧矩阵的一行与右侧矩阵的一列对应相乘后求和。

例如，结果左上角的元素是：

$$0 \times 0 + 1 \times 3 + 2 \times 6 + 3 \times 9 = 42$$

In [ ]:
np.sum(d[0, :] * e[:, 0]), product[0, 0]

对于本节的二维数组，`d.dot(e)` 与 `d @ e` 得到相同的结果。

In [ ]:
d.dot(e)

### 3.3 无法进行矩阵乘法的情况

如果 `f` 的形状也是 `(3, 4)`，则 `d @ f` 不成立：`d` 的列数为 4，`f` 的行数为 3。

这种情况会引发 `ValueError`。计算之前可先通过 `shape` 检查维度，无需通过运行错误代码判断。

In [ ]:
f = np.arange(12).reshape(3, 4)
d.shape[1] == f.shape[0]

## 4. 要点总结

- `reshape()` 改变数组形状，元素总数必须保持不变。
- 普通数组的 `+`、`-`、`*`、`/` 都按元素进行运算，形状需要广播兼容。
- 数学矩阵可以使用二维 `ndarray` 表示。
- `a * b` 是逐元素乘法；二维数组的 `a @ b` 是矩阵乘法。
- 矩阵乘法要求左侧列数等于右侧行数，结果形状为“左侧行数 × 右侧列数”。